# Steering Factory — Colab runner
This notebook clones the GitHub repository, runs the reusable package on Colab disk, and publishes each finalized artifact to Google Drive.

In [ ]:
import os
import sys

REPO_URL = 'https://github.com/UtkarshC99/steering-factory'
REPO_DIR = '/content/steering-factory'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print('Repo already cloned — pulling latest...')
    !git -C {REPO_DIR} pull --ff-only
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
%cd {REPO_DIR}

In [ ]:
%cd /content/steering-factory
!pip -q install -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Write actively to fast Colab disk; finalized runs are mirrored to Drive.
ARTIFACT_ROOT = '/content/steering_factory_artifacts'
PUBLISH_ROOT = '/content/drive/MyDrive/Projects/steering-factory'
MANIFEST = 'manifests/default_colab.yaml'

In [ ]:
!python -m steering_factory prepare-data $MANIFEST --set artifacts.root=$ARTIFACT_ROOT --set artifacts.publish_root=$PUBLISH_ROOT

## Run helper

Each `steering_factory` CLI invocation prints one JSON line: `{"run_id": ..., "artifact_dir": ..., "status": ...}`.
`run_sf` below runs a command, captures that line, raises if the run didn't finish `completed`, and returns the
artifact directory so later cells (`finetune`, `compare`) can chain off of it without you having to copy-paste paths.

In [ ]:
import json
import subprocess

def run_sf(*args):
    """Runs `python -m steering_factory <args>`, streams output live, and
    returns the parsed {run_id, artifact_dir, status} dict from the final
    JSON line. Raises on a non-zero exit or a non-completed run status."""
    cmd = ['python', '-m', 'steering_factory', *args]
    print('$', ' '.join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end='')
        lines.append(line.rstrip('\n'))
    returncode = proc.wait()
    if returncode != 0:
        raise RuntimeError(f'steering_factory {args[0]} exited with code {returncode}')
    payload = None
    for line in reversed(lines):
        try:
            payload = json.loads(line)
            break
        except json.JSONDecodeError:
            continue
    if payload is None:
        raise RuntimeError(f'Could not find a JSON result line in steering_factory {args[0]} output.')
    if payload.get('status') not in (None, 'completed'):
        raise RuntimeError(f"steering_factory {args[0]} finished with status={payload.get('status')!r}: {payload}")
    return payload

## Steering arm

Runs the full extract + evaluate grid (Pillar 1 safety/behavior recipes and Pillar 3 ablation grid) in one artifact.
For faster iteration on extraction methods or layers alone, use `extract` followed by `evaluate --vectors-run <dir>`
instead — see the README's command reference.

In [ ]:
steering_result = run_sf('run', MANIFEST, '--set', f'artifacts.root={ARTIFACT_ROOT}', '--set', f'artifacts.publish_root={PUBLISH_ROOT}')
STEERING_RUN_DIR = steering_result['artifact_dir']
print('\nSteering run:', STEERING_RUN_DIR)

## QLoRA arm

Trains one matched QLoRA adapter per model/recipe steer split, then evaluates each adapter on the same held-out
validation/test examples the steering arm used — required for `compare` to produce a matched quality/cost report.
This is the slowest cell in the notebook; expect it to take longer than the steering run.

In [ ]:
qlora_result = run_sf('finetune', MANIFEST, '--set', f'artifacts.root={ARTIFACT_ROOT}', '--set', f'artifacts.publish_root={PUBLISH_ROOT}')
QLORA_RUN_DIR = qlora_result['artifact_dir']
print('\nQLoRA run:', QLORA_RUN_DIR)

## Compare

Joins the steering run and the QLoRA run on `(model_id, recipe_id)`: the best steering config is selected on
validation only, held-out test quality is reported for both arms, and cost (wall time, labeled-example count,
artifact size, ms/token) is normalized across both. Writes `report.json` and `report.md` under `--output-root`.

In [ ]:
COMPARISON_OUTPUT = f'{ARTIFACT_ROOT}/comparisons'

!python -m steering_factory compare $STEERING_RUN_DIR $QLORA_RUN_DIR --output-root $COMPARISON_OUTPUT

In [ ]:
from IPython.display import Markdown, display

with open(f'{COMPARISON_OUTPUT}/report.md', encoding='utf-8') as f:
    display(Markdown(f.read()))